# Grafo bipartito de cargos y habilidades ESCO

El grafo conecta cargos observados con habilidades mencionadas explícitamente en vacantes. Es una señal exploratoria de proximidad entre perfiles, no una recomendación laboral individual.

In [ ]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd()
if not (RAIZ / 'src').exists(): RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, Markdown
from src.grafos.metricas_grafo import crear_grafo_bipartito, resumen_red, ranking_grado, centralidad_aproximada, similitudes_ocupaciones, comunidades_ocupaciones, habilidades_puente, detalle_transicion, habilidades_caracteristicas
DIR = RAIZ / 'data/procesados/grafos'
faltantes = [p.name for p in [DIR/'nodos.csv', DIR/'aristas.csv', DIR/'manifiesto.json'] if not p.exists()]
if faltantes: raise FileNotFoundError('Primero ejecuta `python -m src.grafos.construir_grafo --repo pxtron/vacantes-colombia --esco-dir data/referencias/esco`. Faltan: ' + ', '.join(faltantes))
nodos = pd.read_csv(DIR/'nodos.csv')
aristas = pd.read_csv(DIR/'aristas.csv')
manifiesto = json.loads((DIR/'manifiesto.json').read_text(encoding='utf-8'))
display(Markdown(f'**Datos:** `{manifiesto["repo_id"]}` @ `{manifiesto["revision_resuelta"]}` · **ESCO {manifiesto["version_esco"]}**'))

## 1. Estructura de la red

In [ ]:
G = crear_grafo_bipartito(nodos, aristas)
display(pd.Series(resumen_red(G), name='valor').to_frame())
display(Markdown('La densidad se calcula como aristas observadas sobre todas las combinaciones cargo–habilidad posibles. Una red dispersa es esperable cuando las habilidades son específicas.'))

In [ ]:
top_skills = ranking_grado(G, 'habilidad', 20)
top_occ = ranking_grado(G, 'ocupacion', 20)
px.bar(top_skills.sort_values('grado'), x='grado', y='nombre', orientation='h', title='Habilidades conectadas con más cargos').show()
px.bar(top_occ.sort_values('grado'), x='grado', y='nombre', orientation='h', title='Cargos conectados con más habilidades').show()
display(Markdown('El grado mide diversidad de conexiones; la fuerza, disponible en las tablas, suma publicaciones y distingue amplitud de frecuencia.'))
display(top_skills); display(top_occ)
display(centralidad_aproximada(G, muestras=200, semilla=42).head(20))
display(Markdown('La betweenness aproximada identifica nodos que conectan zonas de la red; usa muestreo reproducible para evitar un cálculo cuadrático sobre el grafo completo.'))

## 2. Similitud, comunidades y habilidades puente

In [ ]:
similitudes = similitudes_ocupaciones(nodos, aristas, vecinos=10, similitud_minima=.05)
comunidades = comunidades_ocupaciones(similitudes, semilla=42)
puentes = habilidades_puente(aristas, comunidades, nodos)
nombres = nodos.set_index('node_id')['nombre']
pares = similitudes.head(25).copy()
pares['ocupacion_a'] = pares['source'].map(nombres)
pares['ocupacion_b'] = pares['target'].map(nombres)
display(pares[['ocupacion_a','ocupacion_b','similitud_coseno']].style.format({'similitud_coseno':'{:.3f}'}))
display(puentes.head(20))
display(Markdown('El coseno usa prevalencias TF-IDF: premia habilidades compartidas y reduce el peso de las ubicuas. Las habilidades puente aparecen en varias comunidades, pero no implican por sí solas transferibilidad profesional.'))

## 3. Subred de una ocupación

In [ ]:
# Cambia este texto por otra ocupación de `opciones` y vuelve a ejecutar la celda.
opciones = nodos.loc[nodos.tipo.eq('ocupacion'), ['node_id','nombre']].sort_values('nombre')
OCUPACION_SELECCIONADA = opciones.iloc[0]['nombre']
display(opciones.head(30))
occ_id = opciones.loc[opciones.nombre.eq(OCUPACION_SELECCIONADA), 'node_id'].iloc[0]
skills_top = aristas[aristas.source.eq(occ_id)].nlargest(10, 'peso')['target'].tolist()
vecinos = pd.concat([similitudes[similitudes.source.eq(occ_id)].rename(columns={'target':'vecino'}), similitudes[similitudes.target.eq(occ_id)].rename(columns={'source':'vecino'})]).nlargest(5, 'similitud_coseno')['vecino'].tolist()
seleccion = {occ_id, *skills_top, *vecinos}
for vecino in vecinos: seleccion.update(aristas[aristas.source.eq(vecino)].nlargest(5, 'peso')['target'])
sub = G.subgraph(seleccion).copy()
pos = nx.spring_layout(sub, seed=42, weight='peso')
edge_x, edge_y = [], []
for a,b in sub.edges(): edge_x += [pos[a][0],pos[b][0],None]; edge_y += [pos[a][1],pos[b][1],None]
fig = go.Figure(go.Scatter(x=edge_x,y=edge_y,mode='lines',line={'color':'#bbb','width':1},hoverinfo='none'))
for tipo, color in [('ocupacion','#123c63'),('habilidad','#e8b93a')]:
    ns=[n for n,d in sub.nodes(data=True) if d['tipo']==tipo]
    fig.add_trace(go.Scatter(x=[pos[n][0] for n in ns],y=[pos[n][1] for n in ns],mode='markers+text',text=[sub.nodes[n]['nombre'] for n in ns],textposition='top center',name=tipo,marker={'size':16 if tipo=='ocupacion' else 10,'color':color}))
fig.update_layout(title=f'Subred: {OCUPACION_SELECCIONADA}',showlegend=True,height=700,xaxis={'visible':False},yaxis={'visible':False})
fig.show()
display(habilidades_caracteristicas(occ_id, nodos, aristas, n=10).style.format({'puntaje_tfidf':'{:.3f}'}))
display(Markdown('La subred limita habilidades y vecinos para conservar legibilidad; la tabla TF-IDF destaca habilidades frecuentes en este cargo pero menos ubicuas en los demás.'))

## 4. Señal exploratoria de transición

In [ ]:
if vecinos:
    transicion = detalle_transicion(occ_id, vecinos[0], aristas, nodos)
    display(pd.Series(transicion).to_frame('resultado'))
else:
    display(Markdown('No hay una ocupación vecina que supere el umbral de similitud para el perfil seleccionado.'))

## Limitaciones

La extracción inicial reconoce menciones explícitas de ESCO y puede omitir sinónimos locales o confundir usos no contextuales. Los cargos son títulos normalizados, no ocupaciones homologadas. Las comunidades y rutas describen coocurrencia en vacantes y no competencias reales de una persona.